In [ ]:
file_path = "/content/sample_data/vendor_policy.txt"
with open(file_path, 'r') as file:
    vendor_policy_text = file.read()

print("Successfully loaded the policy document")

In [ ]:
import re

def section_based_chunking(text):
    # Regex to find sections starting with '1. ', '2. ', '3. ', '4. ' etc.
    # It captures the entire section header.
    # The pattern is adjusted to capture the header itself.
    pattern = r'(\d+\.\s[A-Za-z]+\s[A-Za-z]+.*)'

    matches = list(re.finditer(pattern, text))
    chunks = []

    # Iterate through each matched section header
    for i, match in enumerate(matches):
        section_header = match.group(0).strip()
        start_index = match.end() # Start of content after the header

        # Determine the end of the current section's content
        if i + 1 < len(matches):
            end_index = matches[i+1].start()
        else:
            end_index = len(text) # Last section goes to the end of the text

        section_content = text[start_index:end_index].strip()

        # Combine header and content, or just use the header if no content follows
        if section_content:
            chunks.append(f"{section_header}\n{section_content}")
        else:
            chunks.append(section_header)

    # Filter out any potentially empty chunks resulting from extra newlines etc.
    return [chunk for chunk in chunks if chunk.strip()]

semantic_chunks = section_based_chunking(vendor_policy_text)

print(f"Total Semantic/Section-based Chunks: {len(semantic_chunks)}\n")

print("--- Semantic/Section-based Chunks ---")
for i, chunk in enumerate(semantic_chunks):
    print(f"--- Section {i+1} (length: {len(chunk)}) ---")
    print(chunk)
    print("\n")

In [ ]:
from huggingface_hub import login
login(token="YOUR_API_KEY")

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_text = embedding_model.encode(semantic_chunks)

### What is FAISS?
FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that would not fit in RAM. It also provides an optional GPU implementation for even faster searches.

In this notebook, we're using FAISS to index our semantic chunks' embeddings, which allows for very fast similarity searches, crucial for retrieving relevant chunks quickly.

#### Types of FAISS Indexes
FAISS offers various index types, each optimized for different use cases. Here, we used `IndexFlatL2`:

*   **`IndexFlatL2` (L2 Distance)**: This index computes the Euclidean (L2) distance between vectors. It is generally used when the concept of 'distance' or 'dissimilarity' is important, meaning that smaller distances indicate higher similarity. It provides exact nearest neighbor searches.

*   **`IndexFlatIP` (Inner Product)**: This index computes the inner product between vectors. Inner product is often used when similarity is defined by the angle between vectors (e.g., cosine similarity, as a normalized inner product). Larger inner product values indicate higher similarity. It also provides exact nearest neighbor searches.

**Choosing between `IndexFlatL2` and `IndexFlatIP`**: The choice depends on how your embeddings are structured and what measure of similarity is most appropriate for your data. For example, if your embeddings are normalized, inner product is equivalent to cosine similarity.

#### Approximate Nearest Neighbor (ANN) Indexes
For very large datasets where exact search becomes too slow or memory-intensive, FAISS provides Approximate Nearest Neighbor (ANN) indexes (e.g., `IndexHNSW`, `IndexIVFFlat`). These indexes trade a small amount of accuracy for significantly faster search times and reduced memory footprint. You would consider using an ANN index when:
*   Your dataset has millions or billions of vectors.
*   Response time is critical, and a slight reduction in recall (finding the absolute best match) is acceptable.

In [ ]:
%%capture
pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

# Ensure `embedding_text` is defined by executing the preceding cells (e.g., cell BJiWAdTFt7ms)
dimension = embedding_text.shape[1] # Dimension of the embeddings
index = faiss.IndexFlatL2(dimension) # Using L2 distance for similarity

# Add the embeddings to the index
index.add(np.array(embedding_text).astype('float32'))

print(f"Number of vectors in the FAISS index: {index.ntotal}")
print("FAISS index created successfully!")

In [ ]:
def search_chunks(query, k=2):
    # Embed the query
    query_embedding = embedding_model.encode([query])
    # Ensure it's a float32 numpy array, reshaped for FAISS
    query_embedding = np.array(query_embedding).astype('float32').reshape(1, -1)

    # Perform the search
    distances, indices = index.search(query_embedding, k)

    # Retrieve the actual chunks
    results = []
    for i in range(k):
        chunk_index = indices[0][i]
        distance = distances[0][i]
        results.append({
            "chunk": semantic_chunks[chunk_index],
            "distance": distance,
            "index": chunk_index
        })
    return results

# Demonstrate the search function
query_text = "How do I submit an invoice and what are the payment terms?"
print(f"Searching for: '{query_text}'\n")

search_results = search_chunks(query_text, k=2)

print("--- Top 2 Nearest Chunks ---")
for i, result in enumerate(search_results):
    print(f"--- Result {i+1} (Distance: {result['distance']:.4f}, Original Index: {result['index']}) ---")
    print(result['chunk'])
    print("\n")

In [ ]:
queries = [
    "What happens if my invoice doesn't have a PO number?",
    "Can I get paid faster than Net 45?",
    "Who handles unresolved payment disputes?"
]

for query in queries:
    print(f"Searching for: '{query}'\n")
    search_results = search_chunks(query, k=2)
    print("--- Top 2 Nearest Chunks ---")
    for i, result in enumerate(search_results):
        print(f"--- Result {i+1} (Distance: {result['distance']:.4f}, Original Index: {result['index']}) ---")
        print(result['chunk'])
        print("\n")
    print("====================================================\n")